# Denetimde Yapay Zekâ — Lokal Dil Modeli ile Açıklama Sınıflandırma

## Dr. Suat ATAN

**İç Denetim Koordinatörlüğü — Uygulamalı Notebook 3**

Önceki notebook'larda sayılarla çalıştık: tutarlar, tarihler, onay süreleri. Ancak denetim verisinin önemli bir kısmı **serbest metindir**: fatura açıklamaları, ödeme emri gerekçeleri, masraf beyanları…

Bu metinleri tek tek okumak zaman alır. Anahtar kelime ile aramak (`"otel"`, `"yemek"`) ise kısa sürede yetersiz kalır; çünkü aynı şey çok farklı kelimelerle yazılabilir.

> **Bu notebook'un sorusu:**
> “Bir dil modeline açıklamayı okutup harcamanın hangi gruba girdiğini söyletebilir miyim? Söylediğine ne kadar güvenebilirim?”

### Bu notebook'ta ne yapacağız?

1. Yapay zekânın denetimde nerede işe yarayıp nerede yaramadığını konuşacağız.
2. Bilgisayarımızda çalışan (lokal) bir dil modeline — **Ollama** üzerinden **Gemma** modeline — bağlanacağız.
3. 10 kayıtlık küçük bir harcama veri seti oluşturacağız.
4. Modele açıklamaları sınıflandırtacağız ve her cevabın **gerekçesini** de isteyeceğiz.
5. Modelin cevaplarını denetçinin beklediği sınıfla karşılaştıracağız.
6. Sınıflandırmayı tutar bilgisiyle birleştirip bir **inceleme listesi** çıkaracağız.

## 0. Eğitim akışı ve yaklaşık süre

Bu notebook yaklaşık **45–60 dakikalık** bir seans için tasarlanmıştır.

| Bölüm | Süre | Cevapladığı denetim sorusu |
|---|---|---|
| Kavramlar | 10 dk | Dil modeli nedir, neden lokal? |
| Kurulum ve bağlantı | 5 dk | Model çalışıyor mu? |
| Veri seti ve kategoriler | 5 dk | Neyi, hangi gruplara ayıracağız? |
| Prompt (talimat) tasarımı | 10 dk | Modele ne söylersek doğru cevap verir? |
| Toplu sınıflandırma | 10 dk | 10 kaydın hepsi hangi gruba düşüyor? |
| Değerlendirme ve tutarla birleştirme | 10–15 dk | Model ne kadar isabetli? Hangi kayıtlara bakmalıyım? |

## 1. Kavramlar: Dil modeli denetçi için ne yapabilir?

**Büyük dil modeli (LLM)**, çok miktarda metinle eğitilmiş ve kendisine verilen metni “anlamlandırıp” cevap üretebilen bir programdır. ChatGPT, Gemini ve Claude bu türün bilinen örnekleridir.

Bizim kullanacağımız model **Gemma**'dır (Google tarafından açık olarak yayımlanan bir model ailesi). `gemma4:2b` ifadesindeki **2b**, modelin yaklaşık 2 milyar parametreye sahip olduğunu, yani **küçük** bir model olduğunu gösterir. Küçük modeller sıradan bir dizüstü bilgisayarda bile çalışabilir.

**Ollama** ise bu modelleri bilgisayarımıza indirip çalıştırmamızı sağlayan ücretsiz bir programdır. Ollama çalıştığında bilgisayarımızda `http://localhost:11434` adresinde küçük bir servis açar; Python bu adrese soru gönderir, cevabı alır.

### Neden “lokal” model?

| Konu | İnternetteki model (bulut) | Lokal model (Ollama) |
|---|---|---|
| Veri gizliliği | Veri kurum dışına çıkar | **Veri bilgisayardan çıkmaz** |
| Maliyet | Kullanım başına ücret olabilir | Ücretsiz |
| İnternet | Gerekir | Gerekmez (model indirildikten sonra) |
| Başarı | Genellikle daha yüksek | Küçük modellerde daha düşük olabilir |

> **Denetim açısından önemli:**
> Denetim verisi çoğu zaman kişisel veri, ticari sır veya gizli bilgi içerir. Bu verileri kurum onayı olmadan internetteki bir yapay zekâ servisine yapıştırmak **veri güvenliği ihlali** olabilir. Lokal model bu riski büyük ölçüde ortadan kaldırır.

### Dil modelinin zayıf yanları

- **Uydurabilir (halüsinasyon):** Emin olmadığında bile kendinden emin bir cevap verebilir.
- **Her seferinde farklı cevap verebilir:** Bu yüzden `temperature = 0` ayarı ile cevabı mümkün olduğunca sabitleyeceğiz.
- **Talimata bağımlıdır:** Soruyu nasıl sorduğumuz, cevabın kalitesini doğrudan etkiler.

> **Altın kural:** Model bir **ön eleme aracıdır**, denetim kanıtı değildir. Modelin “şüpheli” dediği kayda denetçi bakar; kararı denetçi verir.

## 2. Kurulum ve bağlantı kontrolü

### Bu notebook'u çalıştırmadan önce (bir kez yapılır)

1. https://ollama.com adresinden Ollama'yı indirip kurun.
2. Komut satırını (PowerShell / Terminal) açıp modeli indirin:

```
ollama pull gemma4:2b
```

3. Kurulumu kontrol edin:

```
ollama list
```

Listede `gemma4:2b` görünüyorsa hazırsınız.

> **Not:** Bu notebook Google Colab'da değil, **kendi bilgisayarınızda** (VS Code veya Jupyter) çalıştırılmalıdır; çünkü model sizin bilgisayarınızda çalışmaktadır. Ollama'ya ulaşılamazsa notebook, eğitmenin önceden kaydettiği **örnek cevaplarla** devam eder; böylece ders akışı bozulmaz.

### Aşağıdaki hücre ne yapıyor?

| Satır | Görevi |
|---|---|
| `import requests` | Bir web adresine (burada bilgisayarımızdaki Ollama servisine) istek göndermeye yarayan kütüphane |
| `OLLAMA_URL` | Ollama servisinin bilgisayarımızdaki adresi |
| `MODEL` | Kullanacağımız modelin adı. Başka bir model denemek için sadece burayı değiştirmek yeterli |
| `requests.get(.../api/tags)` | Ollama'ya “hangi modeller yüklü?” diye sorar |
| `OLLAMA_HAZIR` | Bağlantı başarılı ve model yüklüyse `True`, değilse `False` olur; ilerideki hücreler buna göre davranır |

In [ ]:
import json
import requests
import pandas as pd

OLLAMA_URL = "http://localhost:11434"
MODEL = "gemma4:2b"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

try:
    cevap = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    yuklu_modeller = [m["name"] for m in cevap.json()["models"]]
    print("Ollama çalışıyor. Yüklü modeller:", yuklu_modeller)
    OLLAMA_HAZIR = MODEL in yuklu_modeller
    if not OLLAMA_HAZIR:
        print(f"\nUYARI: '{MODEL}' yüklü değil. Komut satırında: ollama pull {MODEL}")
except requests.exceptions.RequestException:
    print("Ollama'ya ulaşılamadı. Program açık mı? (Komut satırında: ollama serve)")
    OLLAMA_HAZIR = False

print("\nCanlı modelle mi çalışacağız?", OLLAMA_HAZIR)

## 3. Eğitim için örnek veri seti

Bir kurumun masraf/ödeme kayıtlarından alınmış gibi düşünülen **10 kayıt** oluşturuyoruz.

| Kolon | Anlamı |
|---|---|
| `kayit_no` | Kayıt numarası |
| `aciklama` | Ödemeyi yapan personelin yazdığı serbest metin açıklama |
| `tutar` | Ödeme tutarı (TL) |
| `denetci_beklentisi` | Denetçinin açıklamayı okuyarak verdiği sınıf. **Modele gösterilmeyecek**; sadece modelin başarısını ölçmek için kullanılacak |

Açıklamaları bilerek **gerçek hayattaki gibi** yazdık: eksik bilgiler, aynı şeyin farklı ifadeleri ve birkaç **dikkat çekici** kayıt var.

In [ ]:
veri = pd.DataFrame({
    "kayit_no": range(1, 11),
    "aciklama": [
        "A4 fotokopi kağıdı 20 koli ve toner alımı",
        "Ankara-İzmir uçak bileti, saha denetimi için",
        "İzmir'de 3 gece otel konaklaması (denetim görevi)",
        "Antivirüs yazılımı yıllık lisans yenileme",
        "Hizmet binası klima bakım ve gaz dolumu",
        "Yönetim kurulu toplantısı sonrası akşam yemeği - 14 kişi",
        "Muhasebe sistemine geçiş için dış danışmanlık hizmeti bedeli",
        "Yılbaşı hediye çeki, personelin eşlerine dağıtılmak üzere",
        "Çeşitli giderler",
        "Müdürün özel aracının lastik değişimi",
    ],
    "tutar": [8_450, 6_200, 12_900, 45_000, 18_750, 27_300, 150_000, 60_000, 95_000, 14_800],
    "denetci_beklentisi": [
        "Kırtasiye ve Ofis",
        "Seyahat ve Konaklama",
        "Seyahat ve Konaklama",
        "Bilgi Teknolojileri",
        "Bakım ve Onarım",
        "Temsil ve Ağırlama",
        "Danışmanlık",
        "Şüpheli / Kurumla İlgisiz",
        "Belirsiz",
        "Şüpheli / Kurumla İlgisiz",
    ],
})

veri

## 4. Kategorileri tanımlayalım

Modelden “bunu sınıflandır” demek yetmez; **hangi sınıflar olduğunu ve her birinin ne anlama geldiğini** açıkça söylememiz gerekir. İşe yeni başlayan bir personele iş anlatır gibi düşünün.

| Kategori | Ne zaman seçilir? |
|---|---|
| Kırtasiye ve Ofis | Kağıt, toner, kalem, dosya gibi ofis sarf malzemeleri |
| Seyahat ve Konaklama | Görev amaçlı ulaşım (uçak, otobüs, araç kiralama) ve otel |
| Bilgi Teknolojileri | Yazılım, lisans, bilgisayar, sunucu, BT donanımı |
| Bakım ve Onarım | Kurum binası ve demirbaşlarının bakım, tamir, servis giderleri |
| Temsil ve Ağırlama | Kurum adına verilen yemek, ikram, misafir ağırlama |
| Danışmanlık | Dışarıdan alınan uzmanlık, danışmanlık, eğitim hizmeti |
| Şüpheli / Kurumla İlgisiz | Kişisel nitelikli, kurumun faaliyetiyle ilişkisi görünmeyen harcamalar |
| Belirsiz | Açıklama sınıflandırmaya yetecek bilgi içermiyorsa |

> **Neden “Belirsiz” sınıfı var?**
> Modele bir kaçış kapısı bırakmazsak, bilgi olmasa bile bir sınıf **uydurmak zorunda** kalır. “Çeşitli giderler” gibi bir açıklama denetçi gözünde de belirsizdir; bu zaten başlı başına bir **bulgudur**.

In [ ]:
KATEGORILER = {
    "Kırtasiye ve Ofis": "Kağıt, toner, kalem, dosya gibi ofis sarf malzemeleri",
    "Seyahat ve Konaklama": "Görev amaçlı ulaşım (uçak, otobüs, araç kiralama) ve otel",
    "Bilgi Teknolojileri": "Yazılım, lisans, bilgisayar, sunucu, BT donanımı",
    "Bakım ve Onarım": "Kurum binası ve demirbaşlarının bakım, tamir, servis giderleri",
    "Temsil ve Ağırlama": "Kurum adına verilen yemek, ikram, misafir ağırlama",
    "Danışmanlık": "Dışarıdan alınan uzmanlık, danışmanlık, eğitim hizmeti",
    "Şüpheli / Kurumla İlgisiz": "Kişisel nitelikli, kurumun faaliyetiyle ilişkisi görünmeyen harcamalar",
    "Belirsiz": "Açıklama sınıflandırmaya yetecek bilgi içermiyorsa",
}

for ad, tanim in KATEGORILER.items():
    print(f"- {ad}: {tanim}")

## 5. Prompt (talimat) tasarımı

Modele gönderdiğimiz metne **prompt** denir. İyi bir prompt dört parçadan oluşur:

| Parça | Bizim örneğimizde |
|---|---|
| **Rol** | “Sen deneyimli bir kamu iç denetçisisin.” |
| **Görev** | “Harcama açıklamasını aşağıdaki kategorilerden birine ata.” |
| **Kurallar** | Kategori listesi + “Sadece bu listeden seç, emin değilsen Belirsiz de.” |
| **Çıktı biçimi** | Cevabı JSON formatında ver: `kategori`, `gerekce`, `guven` |

### Neden JSON?

Model serbest konuşursa (“Bence bu bir kırtasiye gideri olabilir çünkü…”) cevabı Python ile tabloya dökmek zordur. **JSON**, cevabı sabit alanlara böler:

```json
{"kategori": "Kırtasiye ve Ofis", "gerekce": "Kağıt ve toner ofis sarfıdır", "guven": "yuksek"}
```

Ollama'ya ayrıca bir **şema** (`format`) vererek modelin yalnızca bu alanlarla ve yalnızca listedeki kategorilerle cevap vermesini zorlayacağız.

> **Denetim izi açısından önemli:**
> Prompt metni, model adı ve ayarlar (`temperature`) çalışma kâğıdına **aynen** kaydedilmelidir. Aksi halde başka bir denetçi aynı sonucu tekrar üretemez.

### Aşağıdaki hücre ne yapıyor?

| Değişken | Görevi |
|---|---|
| `SISTEM_TALIMATI` | Modelin rolünü, görevini ve kurallarını anlatan sabit metin |
| `CEVAP_SEMASI` | Cevabın hangi alanlardan oluşacağını ve `kategori` alanının hangi değerleri alabileceğini tanımlar |
| `"enum"` | “Bu alan sadece şu değerlerden biri olabilir” demektir |

In [ ]:
kategori_listesi = "\n".join(f"- {ad}: {tanim}" for ad, tanim in KATEGORILER.items())

SISTEM_TALIMATI = f"""Sen deneyimli bir kamu iç denetçisisin.
Görevin: Sana verilen harcama açıklamasını okuyup aşağıdaki kategorilerden BİRİNE atamak.

Kategoriler:
{kategori_listesi}

Kurallar:
1. Sadece yukarıdaki listeden bir kategori seç; yeni kategori uydurma.
2. Açıklama yeterli bilgi içermiyorsa "Belirsiz" seç.
3. Harcama kişisel nitelikli görünüyorsa veya kurumla ilişkisi açık değilse "Şüpheli / Kurumla İlgisiz" seç.
4. Gerekçeni tek kısa Türkçe cümleyle yaz.
5. Güven düzeyini "yuksek", "orta" veya "dusuk" olarak belirt.
"""

CEVAP_SEMASI = {
    "type": "object",
    "properties": {
        "kategori": {"type": "string", "enum": list(KATEGORILER.keys())},
        "gerekce": {"type": "string"},
        "guven": {"type": "string", "enum": ["yuksek", "orta", "dusuk"]},
    },
    "required": ["kategori", "gerekce", "guven"],
}

print(SISTEM_TALIMATI)

## 6. Önce tek bir kayıtla deneyelim

Toplu işleme geçmeden önce **tek bir kayıtla** modelin nasıl cevap verdiğine bakmak iyi bir alışkanlıktır. Bir sorun varsa burada fark ederiz.

### Aşağıdaki fonksiyon ne yapıyor?

| Adım | Açıklama |
|---|---|
| `messages` | Modele iki mesaj gönderiyoruz: `system` (kurallar) ve `user` (sınıflandırılacak açıklama) |
| `format` | Cevabın uyması gereken JSON şeması |
| `temperature: 0` | Modelin “yaratıcılığını” kapatır; aynı soruya mümkün olduğunca aynı cevabı verir |
| `think: False` | Gemma 4 cevap vermeden önce uzun uzun “düşünebilir”. Bu basit görev için düşünme adımını kapatıp hızı artırıyoruz |
| `stream: False` | Cevabın kelime kelime değil, tek parça halinde gelmesini sağlar |
| `json.loads(...)` | Modelden gelen JSON metnini Python sözlüğüne (`dict`) çevirir |

> **Not:** Tutarı modele **göndermiyoruz**. Sınıflandırma sadece açıklamaya göre yapılıyor; tutarı daha sonra denetçi olarak biz değerlendireceğiz. Böylece modelin kararı tutardan etkilenmez.

In [ ]:
def modele_sor(aciklama):
    istek = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SISTEM_TALIMATI},
            {"role": "user", "content": f"Harcama açıklaması: {aciklama}"},
        ],
        "format": CEVAP_SEMASI,
        "options": {"temperature": 0},
        "think": False,
        "stream": False,
    }
    yanit = requests.post(f"{OLLAMA_URL}/api/chat", json=istek, timeout=180)
    yanit.raise_for_status()
    ham_metin = yanit.json()["message"]["content"]
    return json.loads(ham_metin)

Ollama'ya ulaşılamıyorsa aşağıdaki **örnek cevaplar** kullanılacak. Bunlar modelin bu kayıtlara verdiği tipik cevaplardır; ders akışının bozulmaması için burada saklıyoruz.

In [ ]:
ORNEK_CEVAPLAR = {
    1: {"kategori": "Kırtasiye ve Ofis", "gerekce": "Fotokopi kağıdı ve toner ofis sarf malzemesidir.", "guven": "yuksek"},
    2: {"kategori": "Seyahat ve Konaklama", "gerekce": "Saha denetimi için alınan uçak bileti görev seyahatidir.", "guven": "yuksek"},
    3: {"kategori": "Seyahat ve Konaklama", "gerekce": "Denetim görevi kapsamında otel konaklamasıdır.", "guven": "yuksek"},
    4: {"kategori": "Bilgi Teknolojileri", "gerekce": "Yazılım lisansı yenilemesi BT gideridir.", "guven": "yuksek"},
    5: {"kategori": "Bakım ve Onarım", "gerekce": "Hizmet binasındaki klimanın bakımıdır.", "guven": "yuksek"},
    6: {"kategori": "Temsil ve Ağırlama", "gerekce": "Toplantı sonrası verilen yemek ağırlama gideridir.", "guven": "yuksek"},
    7: {"kategori": "Danışmanlık", "gerekce": "Dışarıdan alınan danışmanlık hizmeti bedelidir.", "guven": "yuksek"},
    8: {"kategori": "Temsil ve Ağırlama", "gerekce": "Yılbaşı hediyesi ikram ve ağırlama kapsamında değerlendirilebilir.", "guven": "orta"},
    9: {"kategori": "Belirsiz", "gerekce": "Açıklama harcamanın niteliği hakkında bilgi içermiyor.", "guven": "yuksek"},
    10: {"kategori": "Şüpheli / Kurumla İlgisiz", "gerekce": "Müdürün özel aracına ait gider kişisel niteliktedir.", "guven": "yuksek"},
}

In [ ]:
ilk_aciklama = veri.loc[0, "aciklama"]
print("Gönderilen açıklama:", ilk_aciklama)

if OLLAMA_HAZIR:
    sonuc = modele_sor(ilk_aciklama)
else:
    sonuc = ORNEK_CEVAPLAR[1]

print("\nModelin cevabı:")
print(json.dumps(sonuc, ensure_ascii=False, indent=2))

## 7. Tüm kayıtları sınıflandıralım

Şimdi aynı işi 10 kaydın hepsi için bir döngüyle yapıyoruz. Her kaydın cevabını bir listeye ekleyip en sonunda tabloya dönüştüreceğiz.

> **Pratik not:** Küçük bir model bile her kayıt için birkaç saniye harcayabilir. 10 kayıt için sorun değildir; ancak 100.000 kayıtlık bir veri setinde süre ve tutarlılık ayrıca planlanmalıdır.

In [ ]:
sonuclar = []

for _, satir in veri.iterrows():
    if OLLAMA_HAZIR:
        cevap = modele_sor(satir["aciklama"])
    else:
        cevap = ORNEK_CEVAPLAR[satir["kayit_no"]]

    print(f"{satir['kayit_no']:>2}. {satir['aciklama'][:50]:<50} -> {cevap['kategori']}")
    sonuclar.append({"kayit_no": satir["kayit_no"], **cevap})

sonuc_df = pd.DataFrame(sonuclar).rename(columns={"kategori": "model_kategorisi"})
sonuclu = veri.merge(sonuc_df, on="kayit_no")
sonuclu[["kayit_no", "aciklama", "model_kategorisi", "guven", "gerekce"]]

## 8. Model ne kadar isabetli?

Modelin cevabını denetçinin beklediği sınıfla karşılaştıralım. `eslesti` kolonu iki sınıf aynıysa `True` olur.

> **Dikkat:** 10 kayıtla ölçülen başarı oranı **istatistiksel olarak güvenilir değildir**; sadece fikir verir. Gerçek bir uygulamada en az birkaç yüz kayıtlık, denetçilerce etiketlenmiş bir **test seti** ile ölçüm yapılmalıdır.

In [ ]:
sonuclu["eslesti"] = sonuclu["model_kategorisi"] == sonuclu["denetci_beklentisi"]

dogru = sonuclu["eslesti"].sum()
print(f"Doğru sınıflandırılan: {dogru} / {len(sonuclu)}  (başarı oranı: %{100 * dogru / len(sonuclu):.0f})")

sonuclu[["kayit_no", "aciklama", "denetci_beklentisi", "model_kategorisi", "eslesti"]]

### Uyuşmayan kayıtları inceleyelim

Modelin hatalarına bakmak, başarı oranından **daha öğreticidir**. Gerekçeyi okuyarak modelin neden yanıldığını anlamaya çalışın.

In [ ]:
uyusmayan = sonuclu.loc[~sonuclu["eslesti"], ["kayit_no", "aciklama", "denetci_beklentisi", "model_kategorisi", "gerekce"]]

if uyusmayan.empty:
    print("Tüm kayıtlarda model ile denetçi aynı sınıfı seçti.")
else:
    display(uyusmayan)

**Tartışma:** “Yılbaşı hediye çeki, personelin eşlerine dağıtılmak üzere” kaydı için model ne dedi?

- Kelime düzeyinde bakınca “hediye” → ağırlama/ikram gibi görünebilir.
- Denetçi gözüyle bakınca: Personelin **eşleri** kurum adına ağırlanan kişiler değildir; bu harcamanın kurum kaynağından yapılması mevzuata aykırı olabilir.

Model bu kaydı doğru bilse bile şunu unutmayın: Model **bağlamı ve mevzuatı** denetçi kadar iyi bilmez. Modeli iyileştirmenin bir yolu, prompt'a daha açık kurallar eklemektir (bkz. Alıştırmalar).

## 9. Sınıflandırmayı tutarla birleştirelim

Model açıklamaya bakarak **ne** harcandığını söyledi. Denetçi ise **ne kadar** harcandığını da dikkate alır. İkisini birleştirerek bir **inceleme listesi** oluşturalım.

### Kategori bazında harcama özeti

In [ ]:
ozet = (
    sonuclu.groupby("model_kategorisi")
    .agg(kayit_sayisi=("kayit_no", "count"), toplam_tutar=("tutar", "sum"))
    .sort_values("toplam_tutar", ascending=False)
)
ozet["pay_yuzde"] = ozet["toplam_tutar"] / ozet["toplam_tutar"].sum() * 100
ozet

### İnceleme önceliği kuralları

Basit ve açıklanabilir kurallar kullanıyoruz:

| Kural | Neden? |
|---|---|
| Kategori **Şüpheli / Kurumla İlgisiz** ise → **Yüksek** öncelik | Kişisel harcama riski |
| Kategori **Belirsiz** ise → **Yüksek** öncelik | Açıklaması olmayan harcama, belgelendirme eksikliğidir |
| Model ile denetçi beklentisi uyuşmuyorsa → **Orta** öncelik | İnsan gözüyle yeniden bakılmalı |
| Modelin güveni **düşük** veya **orta** ise → **Orta** öncelik | Model kendisi de tam emin değil |
| Tutar 50.000 TL'nin üzerindeyse → bir kademe yükselt | Önemlilik (materiality) |

> **Not:** Gerçek hayatta `denetci_beklentisi` kolonu olmaz; o kolonu burada sadece eğitim amacıyla kullanıyoruz. Gerçek uygulamada “uyuşmazlık” yerine, modelin **düşük güvenli** cevapları ve **örneklemle yapılan insan kontrolü** kullanılır.

In [ ]:
ONEMLILIK_ESIGI = 50_000
ONCELIK_ADI = {3: "Yüksek", 2: "Orta", 1: "Düşük"}

def oncelik_belirle(satir):
    if satir["model_kategorisi"] in ["Şüpheli / Kurumla İlgisiz", "Belirsiz"]:
        puan = 3
    elif (not satir["eslesti"]) or satir["guven"] != "yuksek":
        puan = 2
    else:
        puan = 1

    if satir["tutar"] > ONEMLILIK_ESIGI:
        puan = min(puan + 1, 3)

    return puan

sonuclu["oncelik_puani"] = sonuclu.apply(oncelik_belirle, axis=1)
sonuclu["inceleme_onceligi"] = sonuclu["oncelik_puani"].map(ONCELIK_ADI)

inceleme_listesi = sonuclu.sort_values(["oncelik_puani", "tutar"], ascending=[False, False])
inceleme_listesi[["kayit_no", "aciklama", "tutar", "model_kategorisi", "guven", "inceleme_onceligi"]]

**Yorum:** 150.000 TL'lik danışmanlık kaydına dikkat edin. Açıklaması gayet normal; ancak tutarı yüksek olduğu için öncelik bir kademe yükseldi. Bu kayıt **sözleşme, hakediş ve hizmetin alındığına dair kanıt** ile birlikte incelenmeye değer. Yani model “normal” dese bile **önemlilik** denetçinin kararıdır.

## 10. Çalışma kâğıdına ne yazmalı?

Yapay zekâ destekli bir analizi denetim dosyasına koyarken aşağıdaki bilgiler kayıt altına alınmalıdır:

| Bilgi | Bu notebook'taki değeri |
|---|---|
| Kullanılan model ve sürümü | `gemma4:2b` (Ollama) |
| Çalışma ortamı | Lokal bilgisayar, veri kurum dışına çıkmadı |
| Prompt metni | `SISTEM_TALIMATI` değişkeninin tamamı |
| Ayarlar | `temperature = 0`, JSON şeması ile kısıtlı çıktı |
| Model başarısının nasıl ölçüldüğü | Denetçi etiketli kayıtlarla karşılaştırma |
| İnsan kontrolü | Yüksek öncelikli kayıtların tamamı denetçi tarafından incelendi |

Aşağıdaki hücre bu bilgileri ve sonuçları bir Excel dosyasına kaydeder.

In [ ]:
from datetime import datetime

calisma_bilgisi = pd.DataFrame({
    "bilgi": ["model", "calisma_tarihi", "temperature", "kayit_sayisi", "prompt"],
    "deger": [
        MODEL if OLLAMA_HAZIR else f"{MODEL} (örnek cevaplar kullanıldı)",
        datetime.now().strftime("%Y-%m-%d %H:%M"),
        0,
        len(sonuclu),
        SISTEM_TALIMATI,
    ],
})

with pd.ExcelWriter("03_llm_siniflandirma_sonuclari.xlsx") as yazici:
    inceleme_listesi.to_excel(yazici, sheet_name="sonuclar", index=False)
    calisma_bilgisi.to_excel(yazici, sheet_name="calisma_bilgisi", index=False)

print("Kaydedildi: 03_llm_siniflandirma_sonuclari.xlsx")

## 11. Alıştırmalar

1. **Kuralı güçlendirin:** `SISTEM_TALIMATI` içine şu kuralı ekleyin ve 8 numaralı kaydı tekrar sınıflandırın:
   *“Personel yakınlarına veya kişilere yapılan hediye, bağış ve ödemeler Şüpheli / Kurumla İlgisiz sayılır.”* Modelin cevabı değişti mi?
2. **Kendi açıklamanızı deneyin:** `modele_sor("...")` fonksiyonunu kendi yazacağınız 3 farklı açıklama ile çağırın. Örneğin: *“Kurum aracı için akaryakıt”*, *“Personel servisi kiralama”*, *“Dizi platformu aboneliği”*.
3. **Tutarlılık testi:** Aynı açıklamayı 3 kez gönderin. Cevap hep aynı mı? Ardından `temperature` değerini `0.8` yapıp tekrar deneyin. Ne değişti?
4. **Model karşılaştırması:** Bilgisayarınızda başka bir model varsa (`ollama list`), `MODEL` değişkenini değiştirip başarı oranını karşılaştırın. Büyük model her zaman daha mı iyi? Ne kadar daha yavaş?
5. **Tartışma:** Bu yöntemi 50.000 kayıtlık bir masraf veri setine uygulamak isteseydiniz, model başarısını nasıl ölçerdiniz? Kaç kaydı insan gözüyle kontrol etmek gerekirdi? (İpucu: Notebook 2'deki örnekleme bölümü.)

> **Özet:** Dil modeli, denetçinin serbest metinli kayıtları **hızlıca gruplamasına ve önceliklendirmesine** yardım eder. Ancak modelin cevabı bir **ipucudur**, kanıt değildir. Prompt, model ve ayarlar belgelenmeli; yüksek riskli kayıtlar mutlaka denetçi tarafından incelenmelidir.